In [6]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from dataset import ClockDataset  # Importing the custom dataset
from torchvision import models

## CNN

In [2]:
# ===================== 1. Define the model (CNN) =====================
class DigitalClockNet(nn.Module):
    def __init__(self):
        super(DigitalClockNet, self).__init__()
        
        # First convolutional block: basic feature extraction (edges, corners)
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # Make it 64x64
        )
        
        # Second convolutional block: more complex features
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # Reduce to 32x32
        )
        
        # Third convolutional block: high-level features
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # Reduce to 16x16
        )

        # Fully connected layers for regression
        # Input size: 128 channels * 16 * 16 spatial dimensions
        self.fc = nn.Sequential( 
            nn.Flatten(), # Flatten the tensor
            nn.Linear(128 * 16 * 16, 512), # Fully connected layer 
            nn.ReLU(),
            nn.Dropout(0.5), # Reduce overfitting
            nn.Linear(512, 3) # Output: (h, m, s)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc(x)
        return x # tensor of shape (batch_size, 3)

In [3]:
# ===================== 2. Training function =====================
def train_model(data_dir, num_epochs=10, batch_size=32, learning_rate=0.0001):
    # Setting up device
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")

    # Data transformations and loaders
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
    ])
    
    train_dataset = ClockDataset(data_dir, subset='train', transform=transform)
    test_dataset = ClockDataset(data_dir, subset='test', transform=transform)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Making the model 
    model = DigitalClockNet().to(device)
    
    # Loss Function: MSE (Mean Squared Error) for regression
    criterion = nn.MSELoss()
    
    # Optimizer: Adam optimizer
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_loss = float('inf')
    os.makedirs('checkpoints', exist_ok=True)

    print("Starting training...")
    
    for epoch in range(num_epochs):
        model.train() # Set model to training mode
        running_loss = 0.0
        
        for batch in train_loader:
            images = batch['digital_img'].to(device)
            labels = batch['time_label'].to(device) # Normalized (h, m, s)
            
            # 1. Forward pass
            outputs = model(images)
            
            # 2. Calculate loss
            loss = criterion(outputs, labels)
            
            # 3. Backward pass and optimization
            optimizer.zero_grad() # Clear gradients
            loss.backward() # Backpropagation
            optimizer.step() # Update weights
            
            running_loss += loss.item() # Update running loss

        avg_train_loss = running_loss / len(train_loader) 

        # Calculate test loss
        model.eval()
        test_loss = 0.0
        with torch.no_grad(): # No gradient calculation during evaluation
            for batch in test_loader:
                images = batch['digital_img'].to(device)
                labels = batch['time_label'].to(device)
                outputs = model(images)
                test_loss += criterion(outputs, labels).item()
        
        avg_test_loss = test_loss / len(test_loader)
        
        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Test Loss: {avg_test_loss:.4f}")

        # Save the best model
        if avg_test_loss < best_loss:
            best_loss = avg_test_loss
            torch.save(model.state_dict(), "checkpoints/digital_reader_best.pth")
            print("  -> Saved best model!")

In [5]:
# ===================== 3. Run =====================
if __name__ == "__main__":
    # Start training with specified data directory and epochs 
    train_model(data_dir="../data", num_epochs=100)

Using device: mps
Starting training...
Epoch [1/100], Train Loss: 0.1029, Test Loss: 0.0902
  -> Saved best model!
Epoch [2/100], Train Loss: 0.0923, Test Loss: 0.0879
  -> Saved best model!
Epoch [3/100], Train Loss: 0.0925, Test Loss: 0.0922
Epoch [4/100], Train Loss: 0.0921, Test Loss: 0.0878
  -> Saved best model!
Epoch [5/100], Train Loss: 0.0912, Test Loss: 0.0896
Epoch [6/100], Train Loss: 0.0909, Test Loss: 0.0901
Epoch [7/100], Train Loss: 0.0922, Test Loss: 0.0892
Epoch [8/100], Train Loss: 0.0912, Test Loss: 0.0874
  -> Saved best model!
Epoch [9/100], Train Loss: 0.0893, Test Loss: 0.0869
  -> Saved best model!
Epoch [10/100], Train Loss: 0.0899, Test Loss: 0.0874
Epoch [11/100], Train Loss: 0.0886, Test Loss: 0.0858
  -> Saved best model!
Epoch [12/100], Train Loss: 0.0885, Test Loss: 0.0846
  -> Saved best model!
Epoch [13/100], Train Loss: 0.0866, Test Loss: 0.0867
Epoch [14/100], Train Loss: 0.0847, Test Loss: 0.0818
  -> Saved best model!
Epoch [15/100], Train Loss: 0.

In [8]:
def denormalize_time(tensor_time):
    h = int(tensor_time[0].item() * 23)
    m = int(tensor_time[1].item() * 59)
    s = int(tensor_time[2].item() * 59)
    return f"{h:02d}:{m:02d}:{s:02d}"

def evaluate():
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # Loading the test dataset for evaluation
    test_dataset = ClockDataset("../data", subset='test', transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=True)

    model = DigitalClockNet().to(device) # Load the trained model
    try:
        model.load_state_dict(torch.load("checkpoints/digital_reader_best.pth", map_location=device))
        print("CNN Model loaded successfully!")
    except FileNotFoundError:
        print("Error: Checkpoint not found.")
        return

    model.eval() # Set model to evaluation mode

    print("\n--- Visual Evaluation CNN ---")
    print(f"{'Actual':<10} | {'Predicted':<10} | {'Diff'}")
    print("-" * 35)

    with torch.no_grad(): # No gradient calculation during evaluation
        for i, batch in enumerate(test_loader):
            if i >= 10: break
            
            img = batch['digital_img'].to(device)
            target = batch['time_label'].to(device)
            
            output = model(img)
            
            actual_str = denormalize_time(target[0])
            pred_str = denormalize_time(output[0])
            
            # Calculate loss for reporting
            loss = torch.nn.functional.mse_loss(output, target).item()
            
            print(f"{actual_str:<10} | {pred_str:<10} | {loss:.4f}")

if __name__ == "__main__":
    evaluate()

CNN Model loaded successfully!

--- Visual Evaluation CNN ---
Actual     | Predicted  | Diff
-----------------------------------
09:40:43   | 34:122:154 | 2.2270
19:40:00   | 48:114:143 | 3.0188
05:48:08   | 43:111:150 | 3.1898
16:16:35   | 35:129:154 | 2.7729
13:50:38   | 36:143:117 | 1.7128
02:30:35   | 36:121:154 | 2.8695
12:50:01   | 34:127:149 | 2.9875
23:20:02   | 39:118:148 | 3.1439
18:30:52   | 40:85:152  | 1.5387
11:40:28   | 40:121:61  | 1.2645


## Res-Net

In [9]:
# ===================== 1. Define the ResNet Model =====================
class DigitalClockResNet(nn.Module):
    def __init__(self):
        super(DigitalClockResNet, self).__init__()
        
        # Load a pre-trained ResNet18 model
        # "DEFAULT" weights mean it was trained on ImageNet (millions of images)
        self.model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        # We need to replace the last layer (fc) because ResNet outputs 1000 classes by default.
        # We want it to output 3 numbers (Hour, Minute, Second).
        
        num_features = self.model.fc.in_features # usually 512 for ResNet18
        
        self.model.fc = nn.Sequential(
            nn.Linear(num_features, 256), # Compress features
            nn.ReLU(),
            nn.Dropout(0.3),              # Prevent overfitting [cite: 46]
            nn.Linear(256, 3)             # Output: (h, m, s)
        )

    def forward(self, x):
        return self.model(x)

In [12]:
# ===================== 2. ResNet Training Function =====================
def train_resnet_model(data_dir, num_epochs=20, batch_size=32, learning_rate=0.0001):
    # Setting up device
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")

    # TRANSFORMATIONS: ResNet requires 224x224 and specific normalization
    transform = transforms.Compose([
        transforms.Resize((224, 224)), 
        transforms.ToTensor(),
        # Standard ImageNet normalization (Mean, Std)
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # Load Data
    # Note: Ensure data_dir points to the folder containing 'train' and 'test' folders
    train_dataset = ClockDataset(data_dir, subset='train', transform=transform)
    test_dataset = ClockDataset(data_dir, subset='test', transform=transform)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Initialize ResNet
    model = DigitalClockResNet().to(device)
    
    # Loss and Optimizer
    criterion = nn.MSELoss()
    # Lower learning rate (0.0001) is better for fine-tuning
    optimizer = optim.Adam(model.parameters(), lr=learning_rate) 

    best_loss = float('inf')
    os.makedirs('checkpoints', exist_ok=True)

    print("Starting ResNet training...")
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for batch in train_loader:
            images = batch['digital_img'].to(device)
            labels = batch['time_label'].to(device)
            
            # Forward -> Loss -> Backward
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader) 

        # Validation
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch in test_loader:
                images = batch['digital_img'].to(device)
                labels = batch['time_label'].to(device)
                outputs = model(images)
                test_loss += criterion(outputs, labels).item()
        
        avg_test_loss = test_loss / len(test_loader)
        
        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Test Loss: {avg_test_loss:.4f}")

        # Save Best Model
        if avg_test_loss < best_loss:
            best_loss = avg_test_loss
            # Save with a distinct name so we don't overwrite the CNN
            torch.save(model.state_dict(), "checkpoints/digital_resnet_best.pth")
            print("  -> Saved best ResNet model!")

# Run the training
# Make sure data_dir is correct (e.g., "./data" or "../data")
if __name__ == "__main__":
    train_resnet_model(data_dir="../data", num_epochs=100)

Using device: mps
Starting ResNet training...
Epoch [1/100], Train Loss: 0.0986, Test Loss: 0.0817
  -> Saved best ResNet model!
Epoch [2/100], Train Loss: 0.0460, Test Loss: 0.0233
  -> Saved best ResNet model!
Epoch [3/100], Train Loss: 0.0272, Test Loss: 0.0131
  -> Saved best ResNet model!
Epoch [4/100], Train Loss: 0.0187, Test Loss: 0.0161
Epoch [5/100], Train Loss: 0.0172, Test Loss: 0.0143
Epoch [6/100], Train Loss: 0.0147, Test Loss: 0.0086
  -> Saved best ResNet model!
Epoch [7/100], Train Loss: 0.0145, Test Loss: 0.0097
Epoch [8/100], Train Loss: 0.0130, Test Loss: 0.0081
  -> Saved best ResNet model!
Epoch [9/100], Train Loss: 0.0132, Test Loss: 0.0063
  -> Saved best ResNet model!
Epoch [10/100], Train Loss: 0.0123, Test Loss: 0.0076
Epoch [11/100], Train Loss: 0.0125, Test Loss: 0.0040
  -> Saved best ResNet model!
Epoch [12/100], Train Loss: 0.0123, Test Loss: 0.0065
Epoch [13/100], Train Loss: 0.0120, Test Loss: 0.0063
Epoch [14/100], Train Loss: 0.0106, Test Loss: 0.00

In [13]:
def evaluate_resnet():
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    
    # MUST MATCH TRAINING TRANSFORMS
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    test_dataset = ClockDataset("../data", subset='test', transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=True)

    model = DigitalClockResNet().to(device)
    
    try:
        # Load the ResNet checkpoint
        model.load_state_dict(torch.load("checkpoints/digital_resnet_best.pth", map_location=device))
        print("ResNet Model loaded successfully!")
    except FileNotFoundError:
        print("Error: ResNet Checkpoint not found.")
        return

    model.eval()

    print("\n--- Visual Evaluation (ResNet) ---")
    print(f"{'Actual':<10} | {'Predicted':<10} | {'Diff'}")
    print("-" * 35)

    def denormalize_time(tensor_time):
        # Clamp ensures values don't go below 0 or above 1 due to minor noise
        tensor_time = torch.clamp(tensor_time, 0, 1) 
        h = int(tensor_time[0].item() * 23)
        m = int(tensor_time[1].item() * 59)
        s = int(tensor_time[2].item() * 59)
        return f"{h:02d}:{m:02d}:{s:02d}"

    with torch.no_grad():
        for i, batch in enumerate(test_loader):
            if i >= 10: break
            
            img = batch['digital_img'].to(device)
            target = batch['time_label'].to(device)
            
            output = model(img)
            
            actual_str = denormalize_time(target[0])
            pred_str = denormalize_time(output[0])
            
            loss = torch.nn.functional.mse_loss(output, target).item()
            
            print(f"{actual_str:<10} | {pred_str:<10} | {loss:.6f}")

if __name__ == "__main__":
    evaluate_resnet()

ResNet Model loaded successfully!

--- Visual Evaluation (ResNet) ---
Actual     | Predicted  | Diff
-----------------------------------
12:21:08   | 13:20:07   | 0.000260
05:13:08   | 06:16:10   | 0.000952
04:11:40   | 05:11:42   | 0.000773
13:57:01   | 14:57:02   | 0.000226
09:40:43   | 09:39:42   | 0.000174
21:43:02   | 20:43:03   | 0.000092
15:45:47   | 17:45:49   | 0.004407
08:32:47   | 09:28:50   | 0.001504
02:11:52   | 02:12:55   | 0.001054
01:42:53   | 01:40:54   | 0.000247
